# Deep Research Tool - デモスクリプト

Deep Research Toolの主要機能をデモンストレーションするノートブックです。

## 目次
1. [環境セットアップ](#1-環境セットアップ)
2. [基本リサーチ](#2-基本リサーチ)
3. [高速クロールモード](#3-高速クロールモード)
4. [V2レポート生成（一貫性保証）](#4-v2レポート生成一貫性保証)
5. [V3レポート生成（DOCX-Native）](#5-v3レポート生成docx-native)
6. [フェルミ推定](#6-フェルミ推定)
7. [検索クエリ最適化のデモ](#7-検索クエリ最適化のデモ)
8. [ResearchWarnings](#8-researchwarnings)
9. [マルチエージェント議論（情報収集付き）](#9-マルチエージェント議論情報収集付き)
10. [全機能フルカスタマイズ例](#10-全機能フルカスタマイズ例)

---

**前提条件:**
- `OPENAI_API_KEY` または `ANTHROPIC_API_KEY` 環境変数を設定済み
- `pip install -e .` で依存関係をインストール済み

## 1. 環境セットアップ

In [ ]:
import os
import sys
from pathlib import Path

# プロジェクトルートをパスに追加
project_root = Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# APIキーの確認
openai_key = os.getenv("OPENAI_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

if openai_key:
    print(f"OpenAI API Key: ...{openai_key[-4:]}")
elif anthropic_key:
    print(f"Anthropic API Key: ...{anthropic_key[-4:]}")
else:
    print("WARNING: APIキーが設定されていません")
    print("  export OPENAI_API_KEY='sk-...'")
    print("  export ANTHROPIC_API_KEY='sk-ant-...'")    

## 2. 基本リサーチ

最もシンプルな使い方です。テーマを指定して `run_research()` を呼ぶだけで、
調査計画の作成 → Web検索 → 情報抽出 → レポート生成が自動で実行されます。

In [ ]:
from deep_research_tool import run_research

result = run_research(
    query="日本のEV市場の現状と将来展望",
    provider="openai",          # or "anthropic"
    iterations=3,                # 各セクションの調査反復回数
    output_format="markdown",    # markdown / docx / pdf / html
    output_dir="./output",
    requirements="最新のトレンドと主要企業の動向を含めること",
)

print(f"レポート: {result['report_path']}")
print(f"エビデンスJSON: {result['evidence_json']}")
print(f"エビデンスCSV: {result['evidence_csv']}")
print(f"トークン使用量: {result['token_usage']}")

### 生成されたレポートの確認

In [ ]:
# Markdownレポートを読み込んで表示
from IPython.display import Markdown, display

report_path = result.get("report_path")
if report_path and Path(report_path).exists():
    with open(report_path, "r", encoding="utf-8") as f:
        content = f.read()
    # 先頭3000文字のみ表示
    display(Markdown(content[:3000] + "\n\n... (以下省略)"))
else:
    print("レポートファイルが見つかりません")

## 3. 高速クロールモード

情報収集を並列化して高速にリサーチを行います。
`fast_batch` はトークン効率が良く、多くのケースで推奨されます。

In [ ]:
result_fast = run_research(
    query="再生可能エネルギー市場の最新動向",
    provider="openai",
    iterations=3,
    output_format="markdown",
    crawl_mode="fast_batch",      # fast_batch / fast_parallel / standard
    fast_crawl_workers=10,        # 並列HTTPワーカー数
    fast_crawl_batch_size=5,      # バッチあたりのページ数
)

print(f"レポート: {result_fast['report_path']}")

### FastCrawlerの独立使用

レポート生成なしで情報収集だけを行いたい場合は `FastCrawler` を直接使用できます。

In [ ]:
from deep_research_tool.research.fast_crawler import FastCrawler, EvaluationMode
from deep_research_tool.search import get_search_client
from deep_research_tool.api import get_client

search_client = get_search_client(method="duckduckgo")
llm_client = get_client(provider="openai")

crawler = FastCrawler(
    search_client=search_client,
    llm_client=llm_client,
    evaluation_mode=EvaluationMode.BATCH,
    max_workers=10,
    batch_size=5,
    language="ja",
)

crawl_result = crawler.crawl_and_evaluate(
    queries=["日本 水素エネルギー 2025", "水素ステーション 整備状況"],
    section_context="日本の水素エネルギー戦略",
    max_pages_per_query=5,
    min_relevance_score=0.3,
)

print(f"取得ページ数: {crawl_result.pages_fetched}")
print(f"関連ページ数: {len(crawl_result.pages)}")
print(f"フェッチ時間: {crawl_result.total_fetch_time:.1f}秒")

for page in crawl_result.pages[:3]:
    print(f"\n  [{page.relevance_score:.2f}] {page.title}")
    print(f"    URL: {page.url}")

## 4. V2レポート生成（一貫性保証）

V2は章間の用語統一・コンテキスト引き継ぎ・一貫性チェックを自動で行います。
長文レポートの品質が大幅に向上します。

In [ ]:
result_v2 = run_research(
    query="量子コンピュータの技術動向と実用化展望",
    provider="openai",
    iterations=3,
    output_format="markdown",
    target_pages=10,

    # V2レポート生成の設定
    report_generator_version="v2",
    v2_writing_style="technical",         # formal / business / technical / executive / casual
    v2_target_audience="engineer",         # expert / business / engineer / general / student
    v2_technical_level=4,                  # 1-5
    v2_enable_consistency_check=True,      # 一貫性チェック
    v2_enable_two_phase=True,              # ドラフト→推敲の2段階
    v2_include_glossary=True,              # 巻末に用語集
)

print(f"V2レポート: {result_v2['report_path']}")

## 5. V3レポート生成（DOCX-Native）

V3はDOCX出力をpython-docx APIで直接構築します。
Markdown中間変換を経由しないため、図表の配置や書式がより正確になります。

**V2との違い:**
- V2: コンテンツ生成 → Markdown → DOCX変換
- V3: コンテンツ生成（V2エンジン） → python-docx APIで直接DOCX構築

In [ ]:
result_v3 = run_research(
    query="次世代半導体材料の技術動向",
    provider="openai",
    iterations=3,
    output_format="docx",           # DOCX形式で出力

    # V3レポート生成
    report_generator_version="v3",   # V3を指定
    v2_writing_style="business",     # V2の設定がそのまま利用可能
    v2_target_audience="business",
    v2_technical_level=3,
)

print(f"V3 DOCXレポート: {result_v3['report_path']}")

### V3ジェネレーターの直接使用

In [ ]:
from deep_research_tool.report.v3 import (
    DocxReportGeneratorV3,
    WritingStyle,
    TargetAudience,
)

# V3ジェネレーターの構成を確認
print("DocxReportGeneratorV3")
print(f"  - V2 (ReportGeneratorV2) を継承")
print(f"  - python-docx APIで直接DOCX構築")
print(f"  - 利用可能な文体: {[s.value for s in WritingStyle]}")
print(f"  - 利用可能な読者層: {[a.value for a in TargetAudience]}")

## 6. フェルミ推定

直接データが得られない定量指標を分解木ベースで推定します。
エビデンス自動照合、感度分析、モンテカルロシミュレーション、
低信頼度リーフの再帰的サブ分解に対応しています。

In [ ]:
result_fermi = run_research(
    query="日本のペットフード市場規模を推定",
    provider="openai",
    iterations=3,

    # フェルミ推定を有効化
    fermi_estimation=True,
    fermi_target_metrics=["日本のペットフード市場規模（年間、円）"],
    fermi_enable_sub_decomposition=True,
    fermi_sub_decomposition_max_iterations=3,
    fermi_sub_decomposition_confidence_threshold=0.65,
)

print(f"レポート: {result_fermi['report_path']}")

### フェルミ推定モジュールの独立使用

In [ ]:
from deep_research_tool.estimation.fermi_estimator import (
    FermiEstimator,
    FermiEstimationConfig,
)
from deep_research_tool.evidence.numerical_extractor import NumericalDataStore

# 設定
fermi_config = FermiEstimationConfig(
    enabled=True,
    max_tree_depth=4,
    monte_carlo_iterations=1000,
    include_sensitivity=True,
    enable_sub_decomposition=True,
)

print("FermiEstimationConfig:")
print(f"  max_tree_depth: {fermi_config.max_tree_depth}")
print(f"  monte_carlo_iterations: {fermi_config.monte_carlo_iterations}")
print(f"  enable_sub_decomposition: {fermi_config.enable_sub_decomposition}")
print(f"  confidence_threshold: {fermi_config.confidence_threshold}")

# 実行にはLLMクライアントが必要
# estimator = FermiEstimator(llm_client=llm_client, config=fermi_config, language="ja")
# result = estimator.estimate(
#     target_metric="日本のペットフード市場規模（年間、円）",
#     data_store=NumericalDataStore(research_topic="ペットフード市場"),
#     context="2024年時点の日本国内市場",
# )
# print(result.to_summary(language="ja"))

## 7. 検索クエリ最適化のデモ

2026-02-19〜20で追加された新機能です。

- **Split**: 長い複合クエリを短い個別クエリに自動分割
- **Anchor**: フォローアップクエリがテーマから逸脱しないようにアンカリング

これらは `run_research()` 内部で自動適用されますが、
ここでは `QueryGenerator` のメソッドを直接呼んで動作を確認します。

In [ ]:
from deep_research_tool.research.query_generator import QueryGenerator

# --- Split: 複合クエリの自動分割 ---
print("=" * 60)
print("Split: 複合クエリの自動分割")
print("=" * 60)

test_queries = [
    "炭素繊維 市場規模",                                      # 短い → そのまま
    "炭素繊維メーカー（東レ・帝人・三菱ケミカル等）の生産能力",   # 括弧付き列挙 → 分割
    "IR収集 年度別売上・CF比率・CAPEX・設備投資",               # 中黒区切り → 分割
    "炭素繊維 PAN系、ピッチ系、リサイクル繊維、天然素材由来の比較分析",  # 読点区切り → 分割
]

split_result = QueryGenerator.split_complex_queries(test_queries)

print(f"\n入力: {len(test_queries)}クエリ → 出力: {len(split_result)}クエリ\n")
for q in split_result:
    print(f"  - {q}")

In [ ]:
from deep_research_tool.utils.helpers import extract_content_words_set

# --- Anchor: テーマアンカリング ---
print("=" * 60)
print("Anchor: テーマアンカリング")
print("=" * 60)

research_topic = "炭素繊維の技術動向調査"
topic_words = extract_content_words_set(research_topic)
print(f"\nテーマ: {research_topic}")
print(f"抽出キーワード: {topic_words}")

# テーマ関連クエリ（アンカリング不要）
query_ok = "炭素繊維メーカー 工場視察 ポイント"
q_words_ok = extract_content_words_set(query_ok)
overlap_ok = q_words_ok & topic_words
print(f"\nクエリ: '{query_ok}'")
print(f"  キーワード: {q_words_ok}")
print(f"  テーマとの重複: {overlap_ok}")
print(f"  → {'通過（アンカリング不要）' if overlap_ok else 'アンカリング実施'}")

# テーマ無関係クエリ（アンカリング実施）
query_drift = "リモートインタビュー 対面調査 比較"
q_words_drift = extract_content_words_set(query_drift)
overlap_drift = q_words_drift & topic_words
print(f"\nクエリ: '{query_drift}'")
print(f"  キーワード: {q_words_drift}")
print(f"  テーマとの重複: {overlap_drift}")
if not overlap_drift:
    primary_keyword = max(topic_words, key=len)
    anchored = f"{primary_keyword} {query_drift}"
    print(f"  → アンカリング実施: '{anchored}'")
else:
    print(f"  → 通過（アンカリング不要）")

## 8. ResearchWarnings

パイプライン実行中のフォールバックやエラーを重要度別に収集する仕組みです。
`run_research()` 内部で自動的に使用されますが、直接操作も可能です。

In [ ]:
from deep_research_tool.utils.helpers import ResearchWarnings

# シングルトンインスタンスを取得
warnings = ResearchWarnings.get_instance()

# デモ用に警告を追加
warnings.add("MEDIUM", "image_download", "Failed to download image from https://example.com/chart.png")
warnings.add("LOW", "formatting", "Japanese font not found, falling back to default")
warnings.add("HIGH", "fermi_estimation", "Sub-decomposition failed for leaf node 'annual_cost'")

# 警告一覧を表示
print("収集された警告:")
for w in warnings.get_all():
    print(f"  [{w['severity']:8s}] {w['category']}: {w['message']}")

# Markdown出力（レポート末尾に追記される形式）
print("\n--- Markdown出力 ---")
print(warnings.to_markdown())

## 9. マルチエージェント議論（情報収集付き）

複数のAIエージェントが議論を行うフレームワーク。
`ResearchParticipantAgent` を使うと、各参加者が議論中にWeb検索で根拠を収集します。

In [ ]:
from multi_agent_discussion import run_discussion

# 情報収集付き議論
discussion_result = run_discussion(
    topic="日本における再生可能エネルギーの将来展望",
    provider="openai",
    participant_personas=[
        {"name": "技術担当", "persona": "再生可能エネルギー技術の専門家"},
        {"name": "経済担当", "persona": "エネルギー市場のアナリスト"},
        {"name": "政策担当", "persona": "エネルギー政策の研究者"},
    ],
    enable_search=True,             # Web検索を有効化
    search_config={
        "region": "jp-jp",
        "max_queries_per_turn": 2,
        "max_results_per_query": 5,
    },
    max_rounds=3,
)

print(discussion_result["transcript"][:2000])

## 10. 全機能フルカスタマイズ例

`run_research()` の主要パラメータを網羅した設定例です。
実際の利用時は必要な機能だけを有効にしてください。

In [ ]:
# 注意: 全機能有効時はAPIコストと実行時間が大幅に増加します
# 必要に応じてコメントアウトを解除して実行してください

full_config = dict(
    # === 基本設定 ===
    query="次世代半導体材料の技術動向と市場展望",
    provider="anthropic",
    model="claude-sonnet-4-20250514",
    iterations=5,
    output_format="docx",
    output_dir="./output",
    requirements="SiC、GaN、Ga2O3の3材料を中心に比較分析",

    # === レポート ===
    target_pages=30,
    report_generator_version="v3",        # V3: DOCX-Native
    v2_writing_style="technical",
    v2_target_audience="engineer",
    v2_technical_level=4,
    v2_enable_consistency_check=True,
    v2_enable_two_phase=True,
    v2_include_glossary=True,

    # === 多言語検索 ===
    multilingual=True,
    search_languages=["ja", "en", "zh"],

    # === 高速クロール ===
    crawl_mode="fast_batch",
    fast_crawl_workers=10,
    fast_crawl_batch_size=5,

    # === 図表・数値 ===
    auto_figures=True,
    numerical_extraction=True,
    intelligent_charts=True,

    # === フェルミ推定 ===
    fermi_estimation=True,
    fermi_enable_sub_decomposition=True,

    # === 検証 ===
    enable_verification=True,
)

print("全機能フルカスタマイズ設定:")
for k, v in full_config.items():
    print(f"  {k}: {v}")

print("\n# 実行するには以下のコメントを解除:")
print("# result_full = run_research(**full_config)")

---

## 機能一覧サマリー

| 機能 | パラメータ | 説明 |
|------|-----------|------|
| 基本リサーチ | `query`, `provider`, `iterations` | テーマ指定でWeb調査→レポート生成 |
| 高速クロール | `crawl_mode="fast_batch"` | 並列フェッチ+バッチLLM評価 |
| V2レポート | `report_generator_version="v2"` | 用語統一・一貫性チェック |
| V3レポート | `report_generator_version="v3"` | DOCX直接構築 (NEW) |
| フェルミ推定 | `fermi_estimation=True` | 分解木ベースの定量推定 |
| クエリ最適化 | 自動適用 | 複合クエリ分割+テーマアンカリング (NEW) |
| ResearchWarnings | 自動適用 | パイプライン警告の可視化 (NEW) |
| 多言語検索 | `multilingual=True` | 複数言語での同時検索 |
| 情報収集議論 | `enable_search=True` | Web検索付きマルチエージェント議論 (NEW) |

*Last updated: 2026-02-20*